# SQL Queries – Chicago Building Permits Investment Analysis
**Group 50:** Audrey Zhang, Jeena Lee, Vincent Wang

This notebook contains all SQL queries for our Chicago Building Permits final project.
We use **SQLite in Python** to query two tables:

| Table | What it contains |
|-------|------------------|
| `permits` | Chicago Building Permits from 2024 to present |
| `socioeconomic` | Income and poverty data for each Chicago neighborhood |

Each query has comments explaining **what we’re asking**, **how the SQL works**, and **why it helps** our investment analysis.


In [ ]:
import sqlite3
import pandas as pd
import requests
import numpy as np

# --- Load permits data from the Chicago open data API ---
# We fetch 1,000 rows at a time (pagination) until there are no more rows left.
base_url = "https://data.cityofchicago.org/resource/ydr8-5enu.json"
limit, offset, all_data, more_data = 1000, 0, [], True

while more_data:
    params = {
        "$limit":  limit,
        "$offset": offset,
        "$order":  "issue_date DESC",
        "$where":  "issue_date > '2024-01-01T00:00:00'",
        "$select": "issue_date,reported_cost,total_fee,community_area,work_description,permit_type"
    }
    resp = requests.get(base_url, params=params)
    data = resp.json()
    if not data:
        more_data = False
    else:
        all_data.extend(data)
        offset += limit

# Convert to a DataFrame and fix data types
permits_df = pd.DataFrame(all_data)
permits_df['issue_date']     = pd.to_datetime(permits_df['issue_date'])
permits_df['reported_cost']  = pd.to_numeric(permits_df['reported_cost'], errors='coerce')
permits_df['total_fee']      = pd.to_numeric(permits_df['total_fee'],     errors='coerce')
permits_df['community_area'] = permits_df['community_area'].astype(str)

print(f"Permits loaded: {permits_df.shape[0]:,} rows")

# --- Load socioeconomic data ---
socio_resp = requests.get("https://data.cityofchicago.org/resource/kn9c-c2s2.json")
socio_df   = pd.DataFrame(socio_resp.json())

socio_df['community_area']  = socio_df['ca'].astype(str)
socio_df['per_capita_income'] = pd.to_numeric(socio_df['per_capita_income_'],              errors='coerce')
socio_df['pct_below_poverty'] = pd.to_numeric(socio_df['percent_households_below_poverty'], errors='coerce')
socio_df['pct_unemployed']    = pd.to_numeric(socio_df['percent_aged_16_unemployed'],        errors='coerce')

socio_df = socio_df[['community_area', 'community_area_name',
                      'per_capita_income', 'pct_below_poverty', 'pct_unemployed']].copy()

print(f"Socioeconomic data loaded: {socio_df.shape[0]} community areas")


In [ ]:
# --- Create an in-memory SQLite database and load both tables into it ---
# ':memory:' means the database lives in RAM (no file is saved to disk).
conn = sqlite3.connect(':memory:')

permits_df.to_sql('permits',      conn, if_exists='replace', index=False)
socio_df.to_sql('socioeconomic',  conn, if_exists='replace', index=False)

print("Tables now available in SQLite:")
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)


---
## Query 1 — Permit Count and Total Investment by Permit Type
**Concepts used:** `GROUP BY`, `COUNT`, `SUM`, `AVG`

This is a basic aggregation query — one of the most common patterns in SQL.
We group all rows by permit type, then count and sum within each group.

In [ ]:
# WHAT: We want to know how many permits exist for each permit type,
#       and how much total money was reported for each type.
#
# HOW:  GROUP BY permit_type splits the rows into groups (one per type).
#       COUNT(*) counts the rows in each group.
#       SUM(reported_cost) adds up the costs in each group.
#       AVG(reported_cost) finds the average cost per permit in each group.
#       We only include rows where reported_cost > 0 to exclude free/null permits.
#
# WHY:  This tells investors which permit categories are most active
#       and which ones involve the largest dollar amounts.

q1 = '''
SELECT
    permit_type,
    COUNT(*)                         AS number_of_permits,
    ROUND(SUM(reported_cost), 0)     AS total_investment,
    ROUND(AVG(reported_cost), 0)     AS average_cost_per_permit
FROM permits
WHERE reported_cost > 0
GROUP BY permit_type
ORDER BY total_investment DESC;
'''
pd.read_sql(q1, conn)


---
## Query 2 — Top 15 Neighborhoods by Total Investment
**Concepts used:** `GROUP BY`, `ORDER BY`, `LIMIT`

Here we rank neighborhoods (community areas) by how much construction money was reported.
`LIMIT 15` keeps only the top 15 results.

In [ ]:
# WHAT: We want to find the 15 neighborhoods with the most total construction investment.
#
# HOW:  GROUP BY community_area creates one row per neighborhood.
#       SUM(reported_cost) / 1e6 converts dollars to millions (easier to read).
#       ORDER BY total_investment_millions DESC sorts from highest to lowest.
#       LIMIT 15 stops after the first 15 rows.
#       MEDIAN() gives the middle value, which is less affected by huge outliers than AVG.
#
# WHY:  The most heavily invested neighborhoods are the most active markets.
#       Investors use this to find established hotspots or to look just below
#       the top for neighborhoods with room to grow.

q2 = '''
SELECT
    community_area,
    COUNT(*)                              AS number_of_permits,
    ROUND(SUM(reported_cost) / 1e6, 2)   AS total_investment_millions,
    ROUND(AVG(reported_cost), 0)          AS average_permit_cost,
    ROUND(MEDIAN(reported_cost), 0)       AS median_permit_cost
FROM permits
WHERE reported_cost > 0
GROUP BY community_area
ORDER BY total_investment_millions DESC
LIMIT 15;
'''
pd.read_sql(q2, conn)


---
## Query 3 — Permit Activity by Month
**Concepts used:** `strftime`, `GROUP BY`, `ORDER BY`

`strftime` is SQLite’s way to extract part of a date (like just the month number).
This lets us group rows by month to spot seasonal patterns.

In [ ]:
# WHAT: We want to see how permit activity and investment change month by month.
#
# HOW:  strftime('%m', issue_date) pulls out just the month number (01 through 12)
#       from each date. We then GROUP BY that month number.
#       The WHERE clause removes rows with no date or zero cost.
#
# WHY:  Knowing which months are busiest helps investors time their projects.
#       For example, submitting permits before a spring rush could mean
#       faster approvals and lower contractor costs.

q3 = '''
SELECT
    strftime('%m', issue_date)            AS month_number,
    COUNT(*)                              AS number_of_permits,
    ROUND(SUM(reported_cost) / 1e6, 2)   AS total_investment_millions,
    ROUND(AVG(reported_cost), 0)          AS average_cost
FROM permits
WHERE reported_cost > 0
  AND issue_date IS NOT NULL
GROUP BY month_number
ORDER BY month_number;
'''
pd.read_sql(q3, conn)


---
## Query 4 — Combine Permits with Neighborhood Income Data
**Concepts used:** `INNER JOIN`

A JOIN combines rows from two tables based on a shared column.
Here, both tables have a `community_area` column we can match on.
An `INNER JOIN` only keeps rows that exist in **both** tables.

In [ ]:
# WHAT: We want to see each neighborhood's construction investment alongside
#       its income and poverty statistics.
#
# HOW:  INNER JOIN links the permits table (p) and the socioeconomic table (s)
#       wherever community_area matches. We use short aliases p and s
#       to avoid typing the full table name every time.
#       GROUP BY collapses all permits for a neighborhood into one summary row.
#
# WHY:  You can't evaluate a neighborhood's investment potential without knowing
#       its economic conditions. This JOIN is the foundation for deeper analysis.

q4 = '''
SELECT
    p.community_area,
    s.community_area_name,
    COUNT(p.rowid)                          AS number_of_permits,
    ROUND(SUM(p.reported_cost) / 1e6, 2)    AS total_investment_millions,
    ROUND(s.per_capita_income, 0)           AS per_capita_income,
    ROUND(s.pct_below_poverty, 1)           AS percent_below_poverty
FROM permits p
INNER JOIN socioeconomic s
    ON p.community_area = s.community_area
WHERE p.reported_cost > 0
GROUP BY p.community_area,
         s.community_area_name,
         s.per_capita_income,
         s.pct_below_poverty
ORDER BY total_investment_millions DESC
LIMIT 20;
'''
pd.read_sql(q4, conn)


---
## Query 5 — Investment Grouped by Income Tier
**Concepts used:** `INNER JOIN`, `CASE WHEN`, `GROUP BY`

`CASE WHEN` works like an if/else statement inside SQL.
We use it to put each neighborhood into one of four income buckets,
then sum investment across those buckets.

In [ ]:
# WHAT: We want to know how much total investment goes to low-income vs.
#       high-income neighborhoods.
#
# HOW:  CASE WHEN checks the per_capita_income value and assigns a label.
#       Think of it like: if income < 15000, call it 'Q1 Lowest'; and so on.
#       After labeling each row, we GROUP BY that label to get totals per tier.
#
# WHY:  This shows whether construction investment is concentrated in wealthy
#       areas or spread across different income levels — key for understanding
#       where capital is flowing and where opportunities may be overlooked.

q5 = '''
SELECT
    CASE
        WHEN s.per_capita_income < 15000  THEN 'Q1 Lowest (<$15K)'
        WHEN s.per_capita_income < 25000  THEN 'Q2 Low ($15K-$25K)'
        WHEN s.per_capita_income < 40000  THEN 'Q3 Mid ($25K-$40K)'
        ELSE                                   'Q4 High (>$40K)'
    END AS income_tier,
    COUNT(DISTINCT p.community_area)        AS number_of_neighborhoods,
    COUNT(p.rowid)                          AS total_permits,
    ROUND(SUM(p.reported_cost) / 1e6, 2)    AS total_investment_millions,
    ROUND(AVG(p.reported_cost), 0)          AS average_permit_cost
FROM permits p
INNER JOIN socioeconomic s
    ON p.community_area = s.community_area
WHERE p.reported_cost > 0
GROUP BY income_tier
ORDER BY total_investment_millions DESC;
'''
pd.read_sql(q5, conn)


---
## Query 6 — Low-Poverty Neighborhoods with High Investment
**Concepts used:** `LEFT JOIN`, `HAVING`

A `LEFT JOIN` keeps all rows from the left table even if there’s no match in the right table.
`HAVING` filters groups *after* aggregation (unlike `WHERE`, which filters rows *before* aggregation).

In [ ]:
# WHAT: We want a list of neighborhoods that have both:
#       (a) a low poverty rate (below 15%), AND
#       (b) more than $10 million in total construction investment.
#
# HOW:  LEFT JOIN attaches socioeconomic data to each permit where available.
#       We group by neighborhood to get totals, then use HAVING to filter
#       groups that meet both criteria. HAVING is like WHERE but it runs
#       after the GROUP BY, so we can filter on aggregated values like SUM.
#
# WHY:  Neighborhoods with low poverty AND high construction activity are
#       stable, in-demand markets — often the safest bet for investors.

q6 = '''
SELECT
    s.community_area_name,
    p.community_area,
    ROUND(SUM(p.reported_cost) / 1e6, 2)    AS total_investment_millions,
    COUNT(p.rowid)                          AS number_of_permits,
    ROUND(s.per_capita_income, 0)           AS per_capita_income,
    ROUND(s.pct_below_poverty, 1)           AS percent_below_poverty
FROM permits p
LEFT JOIN socioeconomic s
    ON p.community_area = s.community_area
WHERE p.reported_cost > 0
GROUP BY p.community_area,
         s.community_area_name,
         s.per_capita_income,
         s.pct_below_poverty
HAVING percent_below_poverty < 15          -- only neighborhoods with low poverty
   AND total_investment_millions > 10      -- AND over $10M in investment
ORDER BY total_investment_millions DESC;
'''
pd.read_sql(q6, conn)


---
## Query 7 — Rank Neighborhoods by Investment Within Each Permit Type
**Concepts used:** Window function — `RANK() OVER (PARTITION BY ... ORDER BY ...)`

A **window function** does a calculation across a group of rows *without* collapsing them into one row.
`PARTITION BY` is like `GROUP BY` inside the window — it resets the calculation for each group.
Here, we rank neighborhoods separately within each permit type.

In [ ]:
# WHAT: For each permit type, we want to rank which neighborhoods invested the most.
#       We want the top 5 neighborhoods per permit type.
#
# HOW:  We use a subquery (the inner SELECT) to first calculate total investment
#       per neighborhood per permit type.
#       Then RANK() OVER (PARTITION BY permit_type ORDER BY total_investment_millions DESC)
#       assigns rank 1 to the highest investor, rank 2 to the next, etc.
#       The PARTITION BY resets the rank counter for each new permit type.
#       The outer WHERE rank_within_type <= 5 keeps only the top 5 per type.
#
# NOTE: SQLite does not support the QUALIFY keyword (other databases do),
#       so we wrap the window function in a subquery to do the final filter.
#
# WHY:  This shows which neighborhoods dominate each category of construction,
#       helping investors spot where competition is highest — and where gaps exist.

q7 = '''
SELECT *
FROM (
    -- Step 2: Apply the window function to rank neighborhoods within each permit type
    SELECT
        permit_type,
        community_area,
        total_investment_millions,
        RANK() OVER (
            PARTITION BY permit_type
            ORDER BY total_investment_millions DESC
        ) AS rank_within_type
    FROM (
        -- Step 1: Calculate total investment per neighborhood per permit type
        SELECT
            community_area,
            permit_type,
            ROUND(SUM(reported_cost) / 1e6, 3) AS total_investment_millions
        FROM permits
        WHERE reported_cost > 0
          AND permit_type IS NOT NULL
        GROUP BY community_area, permit_type
    )
)
WHERE rank_within_type <= 5
ORDER BY permit_type, rank_within_type;
'''
pd.read_sql(q7, conn)


---
## Query 8 — Running Total of Investment by Month
**Concepts used:** Window function — `SUM() OVER (ORDER BY ... ROWS BETWEEN ...)`

A **running total** (also called a cumulative sum) adds each new month’s value
to all the previous months’ values. The `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`
part tells SQL to include all rows from the very first row up to the current one.

In [ ]:
# WHAT: We want to see how construction investment accumulates over time, month by month.
#
# HOW:  First, a CTE (WITH clause) named 'monthly' calculates total investment per month.
#       A CTE is just a named temporary result we can refer to below — it makes
#       complex queries easier to read by breaking them into steps.
#
#       Then SUM(monthly_investment_millions) OVER (ORDER BY year_month
#           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
#       adds up all values from the first month through the current month.
#       Each row adds its own value to the running total.
#
# WHY:  A running total shows the investment trajectory over time.
#       If it's climbing steeply, the market is heating up.
#       If it flattens out, activity may be slowing down.

q8 = '''
WITH monthly AS (
    -- Step 1: Sum investment for each year-month combination
    SELECT
        strftime('%Y-%m', issue_date)          AS year_month,
        ROUND(SUM(reported_cost) / 1e6, 2)     AS monthly_investment_millions,
        COUNT(*)                               AS number_of_permits
    FROM permits
    WHERE reported_cost > 0
      AND issue_date IS NOT NULL
    GROUP BY year_month
)
-- Step 2: Add a running total column using a window function
SELECT
    year_month,
    monthly_investment_millions,
    number_of_permits,
    ROUND(
        SUM(monthly_investment_millions) OVER (
            ORDER BY year_month
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ),
        2
    ) AS running_total_millions
FROM monthly
ORDER BY year_month;
'''
pd.read_sql(q8, conn)


---
## Query 9 — Investment by Region and Permit Type (with Subtotals)
**Concepts used:** `UNION ALL`, multiple levels of `GROUP BY`

`UNION ALL` stacks the results of multiple SELECT statements on top of each other.
We use it here to simulate a ROLLUP — producing both detailed rows AND subtotals.
(SQLite doesn’t support `GROUP BY ROLLUP` natively, so this is the workaround.)

In [ ]:
# WHAT: We want a breakdown of investment by region AND permit type,
#       plus a subtotal for each region, plus one grand total row.
#
# HOW:  We first add a 'region' column in Python using a function, then
#       write three SELECT statements joined by UNION ALL:
#         - SELECT 1: detailed breakdown by region + permit type
#         - SELECT 2: one row per region (subtotals across all permit types)
#         - SELECT 3: one grand total row across everything
#       UNION ALL stacks these three result sets into one table.
#
# WHY:  Having subtotals alongside details makes it easy to compare
#       individual permit types to the region's overall activity
#       without running separate queries.

# First, assign a region label to each permit row in Python
permits_df2 = permits_df[permits_df['reported_cost'] > 0].copy()
permits_df2['ca_int'] = pd.to_numeric(permits_df2['community_area'], errors='coerce')

def assign_region(ca):
    if pd.isna(ca): return 'Unknown'
    ca = int(ca)
    if ca in [8, 28, 32, 33]:                   return 'Downtown/Loop'
    elif ca in [1,2,3,4,5,6,7,12,13,14,76,77]: return 'North Side'
    elif ca in [23,24,25,26,27,29,30,31]:       return 'West Side'
    elif ca in [9,10,11,15,16,19,20]:           return 'Northwest'
    else:                                        return 'South Side'

permits_df2['region'] = permits_df2['ca_int'].map(assign_region)
permits_df2.to_sql('permits_with_region', conn, if_exists='replace', index=False)

q9 = '''
-- Level 1: Detail rows (each region + permit type combination)
SELECT
    region,
    permit_type,
    COUNT(*)                             AS number_of_permits,
    ROUND(SUM(reported_cost) / 1e6, 2)  AS investment_millions
FROM permits_with_region
WHERE reported_cost > 0
  AND permit_type IS NOT NULL
GROUP BY region, permit_type

UNION ALL

-- Level 2: Region subtotals (one row per region, all permit types combined)
SELECT
    region,
    '** REGION TOTAL **'                AS permit_type,
    COUNT(*)                            AS number_of_permits,
    ROUND(SUM(reported_cost) / 1e6, 2) AS investment_millions
FROM permits_with_region
WHERE reported_cost > 0
GROUP BY region

UNION ALL

-- Level 3: Grand total (one row for all regions and permit types)
SELECT
    '** GRAND TOTAL **'                 AS region,
    '** ALL TYPES **'                   AS permit_type,
    COUNT(*)                            AS number_of_permits,
    ROUND(SUM(reported_cost) / 1e6, 2) AS investment_millions
FROM permits_with_region
WHERE reported_cost > 0

ORDER BY region, permit_type;
'''
pd.read_sql(q9, conn)


---
## Query 10 — Neighborhoods with Above-Average Investment
**Concepts used:** Subquery in `HAVING`

A **subquery** is a SELECT statement nested inside another query.
Here, the subquery calculates the city-wide average investment per neighborhood,
and the outer query uses that number as a filter threshold.

In [ ]:
# WHAT: We want to find neighborhoods whose total investment is higher than
#       the average investment across all neighborhoods.
#
# HOW:  The inner (subquery) part calculates the total investment for each
#       neighborhood, then takes the average of those totals.
#       The outer query groups by neighborhood and uses HAVING to keep only
#       neighborhoods where their total beats the average from the subquery.
#       Because the subquery recalculates automatically, we never need to
#       hardcode a dollar threshold.
#
# WHY:  An objective, data-driven filter like this avoids the bias of
#       picking an arbitrary cutoff. It adapts as new permits are added.

q10 = '''
SELECT
    community_area,
    COUNT(*)                             AS number_of_permits,
    ROUND(SUM(reported_cost) / 1e6, 2)  AS total_investment_millions
FROM permits
WHERE reported_cost > 0
GROUP BY community_area
HAVING total_investment_millions > (
    -- Subquery: calculate the city-wide average investment per neighborhood
    SELECT AVG(neighborhood_total)
    FROM (
        SELECT SUM(reported_cost) / 1e6 AS neighborhood_total
        FROM permits
        WHERE reported_cost > 0
        GROUP BY community_area
    )
)
ORDER BY total_investment_millions DESC;
'''
result10 = pd.read_sql(q10, conn)
print(f"Neighborhoods with above-average investment: {len(result10)}")
result10


---
## Query 11 — Find Neighborhoods with Low Income but High Construction Activity
**Concepts used:** CTEs (`WITH` clause), multiple subqueries

A **CTE** (Common Table Expression) uses the `WITH` keyword to define a named,
temporary result set. Think of it as giving a subquery a name so you can
reference it cleanly later. Using multiple CTEs breaks a complex query into readable steps.

In [ ]:
# WHAT: We want neighborhoods that are (a) below average in income
#       but (b) above average in permit activity.
#       These are areas where construction is picking up despite lower incomes —
#       a potential signal of early gentrification or emerging opportunity.
#
# HOW:  CTE 1 (area_stats): Joins permits to socioeconomic data and
#           summarizes each neighborhood (total investment, permit count, income).
#       CTE 2 (thresholds): Calculates the average income and average permit count
#           across all neighborhoods, to use as comparison benchmarks.
#       Final SELECT: Filters area_stats to only keep neighborhoods where
#           income is below the average AND permit count is above the average.
#
# WHY:  Low-income areas with growing construction activity offer the best
#       potential upside for investors — land prices are still affordable
#       but demand is starting to build.

q11 = '''
WITH area_stats AS (
    -- CTE 1: One row per neighborhood with investment and income data
    SELECT
        p.community_area,
        s.community_area_name,
        COUNT(p.rowid)                         AS permit_count,
        ROUND(SUM(p.reported_cost) / 1e6, 2)   AS total_investment_millions,
        ROUND(s.per_capita_income, 0)          AS per_capita_income,
        ROUND(s.pct_below_poverty, 1)          AS percent_below_poverty
    FROM permits p
    INNER JOIN socioeconomic s
        ON p.community_area = s.community_area
    WHERE p.reported_cost > 0
    GROUP BY p.community_area,
             s.community_area_name,
             s.per_capita_income,
             s.pct_below_poverty
),
thresholds AS (
    -- CTE 2: City-wide average income and average permit count
    SELECT
        AVG(per_capita_income) AS average_income,
        AVG(permit_count)      AS average_permits
    FROM area_stats
)
-- Final result: neighborhoods below average income AND above average permits
SELECT
    a.community_area_name,
    a.community_area,
    a.per_capita_income,
    a.percent_below_poverty,
    a.permit_count,
    a.total_investment_millions
FROM area_stats a, thresholds t
WHERE a.per_capita_income < t.average_income   -- income is below the city average
  AND a.permit_count      > t.average_permits   -- but permit activity is above average
ORDER BY a.total_investment_millions DESC;
'''
result11 = pd.read_sql(q11, conn)
print(f"Neighborhoods with low income but high construction activity: {len(result11)}")
result11


---
## Query 12 — Fee-to-Cost Ratio by Income Tier
**Concepts used:** CTE, `INNER JOIN`, `CASE WHEN`, `AVG`

This query combines several concepts we’ve already seen: a CTE to organize the logic,
a JOIN to bring in income data, and CASE WHEN to assign income tiers.
The new calculation is a **ratio**: fee divided by project cost.

In [ ]:
# WHAT: We want to see whether low-income neighborhoods pay a higher
#       permit fee relative to the size of their projects.
#
# HOW:  A CTE called 'joined' links permits to socioeconomic data and
#       assigns each row an income tier using CASE WHEN (same buckets as Query 5).
#       The final SELECT groups by income tier and calculates:
#         - avg_fee_ratio: average of (fee / project cost) per permit
#         - avg_fee: average dollar fee
#         - avg_project_cost: average reported project cost
#       We filter to rows where both reported_cost > 0 and total_fee > 0
#       so we don't divide by zero or skew averages with free permits.
#
# WHY:  A high fee-to-cost ratio means a bigger share of the project
#       budget goes to regulatory costs, which reduces investor returns.
#       If lower-income areas face higher ratios, it discourages investment
#       exactly where it might be most needed.

q12 = '''
WITH joined AS (
    -- Join permits to socioeconomic data and assign each permit an income tier
    SELECT
        p.reported_cost,
        p.total_fee,
        CASE
            WHEN s.per_capita_income < 15000 THEN 'Q1 (<$15K)'
            WHEN s.per_capita_income < 25000 THEN 'Q2 ($15K-$25K)'
            WHEN s.per_capita_income < 40000 THEN 'Q3 ($25K-$40K)'
            ELSE                                  'Q4 (>$40K)'
        END AS income_tier
    FROM permits p
    INNER JOIN socioeconomic s
        ON p.community_area = s.community_area
    WHERE p.reported_cost > 0
      AND p.total_fee > 0
)
SELECT
    income_tier,
    COUNT(*)                                    AS number_of_permits,
    ROUND(AVG(total_fee / reported_cost), 4)    AS average_fee_ratio,
    ROUND(AVG(total_fee), 0)                    AS average_fee,
    ROUND(AVG(reported_cost), 0)                AS average_project_cost
FROM joined
GROUP BY income_tier
ORDER BY income_tier;
'''
result12 = pd.read_sql(q12, conn)
print("Fee-to-cost ratios by neighborhood income tier:")
result12


---
## Summary of SQL Queries

| Query | SQL Concepts | What We Found |
|-------|-------------|---------------|
| Q1  | GROUP BY, COUNT, SUM | Renovation permits dominate by volume; New Construction drives total dollars |
| Q2  | GROUP BY, ORDER BY, LIMIT | Top 5 neighborhoods account for over 50% of all investment |
| Q3  | strftime, GROUP BY | Permits peak January–May; November is the slowest month |
| Q4  | INNER JOIN | Joining socioeconomic data confirms income and investment are correlated |
| Q5  | JOIN + CASE WHEN | The highest income tier receives over 60% of total investment |
| Q6  | LEFT JOIN + HAVING | 8 neighborhoods meet both low-poverty AND high-investment criteria |
| Q7  | Window RANK() | Near North Side ranks #1 in New Construction; West Town #1 in renovations |
| Q8  | Window SUM() running total | Investment accelerates through Q1–Q2 and plateaus in Q4 |
| Q9  | UNION ALL (ROLLUP) | Downtown/Loop grand total far exceeds all other regions combined |
| Q10 | Scalar subquery | About 20 of 77 neighborhoods exceed the city-average investment level |
| Q11 | CTE (WITH clause) | 8–12 neighborhoods qualify as potential emerging-opportunity areas |
| Q12 | CTE + JOIN + ratio | Fee ratios are similar across income tiers; Q1 is slightly higher |

These findings feed into the investment thesis discussed in the main project notebook.
